# Lesson 35 · Euler meets the FFT — a second hand on the unit circle: forging the FFT's twiddle factor from Euler's formula yourself

**Today's goal**: first see how Euler's formula turns an "angle" into "a point in the complex plane", then bring that relationship down to `twiddle(k, n, N)` and write the twiddle factor the FFT actually uses.

> **Review of L06 + DSP conventions**: Euler's formula was covered in L06. This lesson forges it into a callable function using the DFT indices k, n, N; the **minus sign** follows the DFT definition — don't change it on a whim. No need for a third full proof of Euler.

When L37–L39 rebuild the FFT, every line calls `twiddle`; write it well today and reuse it straight away later.

← **Previous lesson** · [L34 · Nyquist and aliasing](https://github.com/djzoom/AURORA/blob/main/notebooks/5_audio_dsp/L34_aliasing.ipynb) *(Chinese)*

> Last lesson covered **Nyquist and aliasing**: what a 6 kHz sine wave becomes when sampled at 8 kHz.
> This lesson explores **Euler meets the FFT**.

## Learning goals

By the end of this lesson you should be able to:

1. **Implement `euler(theta)`**: build `e^{iθ}` from first principles with `np.cos` and `np.sin`, without relying on `np.exp`.
2. **Implement `twiddle(k, n, N)`**: call `euler` to compute the FFT twiddle factor `e^{-2πikn/N}`, and explain the physical meaning of the minus sign (clockwise / the analysis direction).
3. **Pass the assertions**: make every assert in the check cell pass, including the non-trivial case `twiddle(1,2,8) == -1j`.
4. **Connect to the Aurora source**: find the place in `src/aurora/audio/transforms.py` that uses an equivalent twiddle-factor matrix.

## Review bridge · L07 square wave = a sum of harmonics (2 minutes)

L07 showed that a periodic waveform can be written as a weighted sum of sine waves — a square wave ≈ the sum of odd harmonics `sin(2πkt)/k`.

The DFT/FFT solves the **inverse problem**: given a stretch of signal, ask "how big is each frequency component?"
The `twiddle(k, n, N)` you write today is the rotation position at discrete time step `n` for frequency `k`.

Carry this comparison into Euler's formula: the complex exponential is nothing new — it's sin and cos merged into a coordinate on the unit circle.


## This lesson's story: the rotating code

The FFT's core formula contains a mysterious exponential: `e^{-2πikn/N}`. It isn't an ordinary real exponential but a **complex number rotating on the unit circle**.

**Euler's formula**:
$$e^{i\theta} = \cos\theta + i\sin\theta$$

Intuition: picture the second hand of a watch, pinned to the unit circle. The angle θ tells it where to point — the horizontal coordinate is cos θ, the vertical coordinate is sin θ. Merge those two coordinates into one complex number, and you have `e^{iθ}`.

**Why does this formula hold?** (no need to memorize it — just get the idea)

From the Taylor series, we know:
$$e^x = 1 + x + \frac{x^2}{2!} + \frac{x^3}{3!} + \ldots$$

Let `x = iθ` (imaginary) and substitute:
$$e^{i\theta} = 1 + i\theta + \frac{(i\theta)^2}{2!} + \frac{(i\theta)^3}{3!} + \ldots$$

Note that `i² = -1`, `i³ = -i`, `i⁴ = 1`… so the even and odd terms separate:
- The even terms (1, -θ²/2!, θ⁴/4!, …) are exactly the Taylor series of `cos θ`
- The odd terms (iθ, -iθ³/3!, …) are exactly the Taylor series of `i·sin θ`

So `e^{iθ} = cos θ + i·sin θ` isn't a definition pulled out of thin air; it's a deep link between the exponential function and the trig functions.

**The geometry of complex multiplication (the polar view)**:

When two complex numbers are multiplied in polar form (magnitude r, argument θ):
$$(r_1 e^{i\theta_1}) \times (r_2 e^{i\theta_2}) = (r_1 r_2) e^{i(\theta_1 + \theta_2)}$$

In words: **magnitudes multiply, arguments add.** That's why multiplying by a point on the unit circle (a complex number of magnitude 1) only rotates and never scales — the product of the magnitudes is still 1.

**Why does the FFT need it?**

The k-th output of the DFT (discrete Fourier transform):

$$X[k] = \sum_{n=0}^{N-1} x[n] \cdot e^{-2\pi i k n/N}$$

Each `e^{-2πikn/N}` is a **twiddle factor**: the position the second hand reaches at step n when it rotates at frequency k. Multiplying the signal x[n] by this twiddle factor amounts to asking "how big is the projection of x[n] onto this frequency?"

**What does "projection" actually mean? (an everyday analogy)**

Imagine you're in a dark room holding a flashlight, shining it in a fixed direction at a small ball swinging back and forth — the flashlight's direction is "the rotation direction for frequency k". How long a shadow the ball casts along that direction is the "projection": if the ball swings exactly along the beam, the shadow is long (lots of energy at this frequency); if the two are perpendicular, the shadow shrinks almost to a point (almost no energy at this frequency).

What the DFT does is, for every candidate frequency k, "shine a beam in a particular direction that also rotates over time" (that is, multiply by the twiddle factor `e^{-2πikn/N}`), then add up the "shadows" of all the samples (sum over n — the Σ in the formula). If the signal really contains a component at frequency k, the shadows keep piling up in roughly the same direction, and the sum is a complex number with a large magnitude; if the signal has no such frequency, shadows pointing in opposite directions cancel out and the sum is close to 0. That's the intuitive meaning of "how big the projection is" — **the more it accumulates, the more of the signal is "hiding" at this frequency**.

This section implements two functions:
- `euler(theta)` → `cos(theta) + i·sin(theta)` (Euler's formula)
- `twiddle(k, n, N)` → `e^{-2πi·k·n/N}` (the FFT twiddle factor)

## 1. First, treat complex numbers as points in the plane

- The complex number `a + b·i` corresponds to the point `(a, b)` in the plane.
- The real part `a` is the horizontal coordinate; the imaginary part `b` is the vertical coordinate.
- Multiplying by `i` rotates the point 90° counterclockwise.
- `e^{iθ}` is the point on the unit circle at angle `θ`, namely `(cos θ, sin θ)`.

**Why is the point on the unit circle at angle θ exactly (cos θ, sin θ)?**

Picture a line segment from the origin to a point on the unit circle, making an angle θ with the positive x-axis. From that point drop a perpendicular down to the x-axis and another across to the y-axis: the perpendicular to the y-axis has length cos θ and the one down to the x-axis has length sin θ — that's just the right-triangle definition of the trig functions. So writing this point as a complex number naturally gives `cos θ + i·sin θ`.

**Why does complex multiplication rotate?**

Write two complex numbers in polar form:
- `z₁ = r₁·e^{iθ₁}` is the complex number with magnitude r₁ and argument θ₁
- `z₂ = r₂·e^{iθ₂}` is the complex number with magnitude r₂ and argument θ₂

Multiply them:
$$z_1 \times z_2 = r_1 e^{i\theta_1} \times r_2 e^{i\theta_2} = (r_1 r_2) \cdot e^{i(\theta_1 + \theta_2)}$$

The result: the magnitude becomes `r₁×r₂` and the argument becomes `θ₁+θ₂`. In plain words: **"rotate by θ₂, then scale by r₂"**.

In particular, if z₂ is on the unit circle (r₂ = 1), it **only rotates and never scales**. That's why the FFT multiplies the signal by complex numbers on the unit circle (twiddle factors) — they change only the phase, not the amplitude.

**Why assemble cos+i·sin by hand instead of just writing `np.exp(1j*theta)`?** The two give exactly the same numbers, but building the real and imaginary parts yourself lets you see Euler's formula hold with your own eyes — instead of treating it as a black-box function call. When L37–L39 rebuild the FFT, `twiddle` is called once for every pair `(k, n)`, and N² calls piece together the full spectrum; understand the underlying construction today and next week you'll be able to read what every row of the matrix is doing.

## 1.1 Complex multiplication, starting with the 90° rotation

Before any formula, just look at `z * 1j`. The imaginary unit `1j` sits on the unit circle at argument π/2 (90°). By the polar rule "arguments add", multiplying any complex number by `1j` adds π/2 to its argument — a 90° counterclockwise rotation.

Every multiplication by `1j` turns the point 90° counterclockwise around the origin. Check it with the code below:

### 🤔 Why use imaginary numbers instead of a vector pair (cos θ, sin θ)?

You might ask: since cos θ and sin θ already describe a point on the unit circle, why insist on the complex number `cos θ + i·sin θ`? Wouldn't the real pair `(cos θ, sin θ)` do just as well?

**The answer: imaginary numbers turn rotation into multiplication.**

- With a **vector pair**: rotation needs a matrix — to rotate the point `(x, y)` by angle θ you need a rotation matrix, a 2×2 operation.
- With a **complex number**: rotation is just multiplication — a point `z = x + iy` times `e^{iθ}` is automatically rotated by θ, no matrix needed.

**The geometric meaning of the imaginary unit `i` is "rotate 90° counterclockwise".** Look:
- `1 × i = i` (the point `(1,0)` times i becomes `(0,1)` — rotated 90°)
- `i × i = -1` (the point `(0,1)` times i again becomes `(-1,0)` — another 90°, 180° in total)

That's why the FFT uses complex numbers rather than vector pairs: **rotation happens over and over, and multiplication is far tidier than matrices.**

In [ ]:
# Aurora matplotlib bootstrap
from pathlib import Path
import sys

_root = None
_cwd = Path.cwd().resolve()
for _candidate in (_cwd, *_cwd.parents):
    if (_candidate / '_matplotlib_bootstrap.py').exists():
        _root = _candidate
        break
if _root is None:
    _notebooks_dir = _cwd / 'notebooks'
    if _notebooks_dir.exists():
        for _found in _notebooks_dir.rglob('_matplotlib_bootstrap.py'):
            _root = _found.parent
            break
if _root is not None and str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from _matplotlib_bootstrap import apply as _aurora_mpl_apply
_aurora_mpl_apply()


In [ ]:
import numpy as np

z = 1 + 0j
for step in range(5):
    print(f'step={step}  z={z.real:+.0f}{z.imag:+.0f}j')
    z = z * 1j


## Into the lab: looking at a sequence of angles

These experiments take angles spread evenly over [0, 2π) as input and line up cos, sin and the complex coordinate side by side, confirming that all three always point to the same spot on the unit circle.

In [ ]:
import numpy as np
print('Ready; imaginary unit example:', 1j * 1j)  # = (-1+0j)

## Hands-on: from samples to a sequence of angles

The cell below takes a sine wave apart for you: sample index `n` → time `t` → angle on the circle `angle = 2π·f·t` → waveform value `sin(angle)`. Adjust `freq` or `sample_rate` and watch how the angle sequence changes — these angles are exactly the inputs we'll feed into Euler's formula shortly.

In [ ]:
import numpy as np

sample_rate = 8
duration = 1.0
freq = 2.0
N = round(duration * sample_rate)
n = np.arange(N)
t = n / sample_rate
angle = 2 * np.pi * freq * t
wave = np.sin(angle)

print('N =', N)
print('sample index n =', n)
print('time axis t =', np.round(t, 3))
print('angle =', np.round(angle, 3))
print('sin(angle) =', np.round(wave, 3))


## Code experiment: lining up angles, trig functions and complex coordinates

Print three columns side by side for the same batch of angles — cos, sin and exp(1j·θ) — and check that the real part of exp(1j·θ) equals cos θ and its imaginary part equals sin θ.

In [ ]:
import numpy as np

angles = np.linspace(0, 2*np.pi, 9)
for theta in angles:
    z = np.exp(1j * theta)
    print(f'theta={theta:5.2f} | cos={z.real:6.3f} | sin={z.imag:6.3f} | z={z.real:6.3f}{z.imag:+6.3f}j')


## 2. ✏️ Implement `euler(theta)`

**Work out these special angles by hand** (the key points on the unit circle):

| theta (radians) | cos θ | sin θ | e^{iθ} |
|---|---|---|---|
| 0 | 1 | 0 | **1+0j** |
| π/2 ≈ 1.5708 | 0 | 1 | **0+1j** (turned 90° counterclockwise) |
| π ≈ 3.1416 | -1 | 0 | **-1+0j** (turned 180°, the hand points left) |
| 3π/2 | 0 | -1 | **0-1j** |
| 2π | 1 | 0 | **1+0j** (one full turn, back to the start) |

**Key property**: |e^{iθ}| = √(cos²θ + sin²θ) = **1** (always on the unit circle)

> **Implementation hint**: `return np.cos(theta) + 1j * np.sin(theta)`

### Before writing code, fill in the variable table

Be clear on three things before writing `euler`:
- Input: `theta` (radians, a scalar or an array)
- Key steps: compute `np.cos(theta)` as the real part and `np.sin(theta)` as the imaginary part, then add them into a complex number
- Return: a complex number (or array of complex numbers) with magnitude 1 and argument theta

**What does NumPy do when you multiply `1j` by an array?**

`1j` is a complex scalar (real part 0, imaginary part 1). When you write `1j * np.sin(theta)` and `theta` is an array (say, 9 angles), NumPy triggers **broadcasting**: the scalar `1j` is "copied" to every position in the array and multiplied element by element, giving a new complex array — element i is `1j * sin(theta[i])`.

Then add `np.cos(theta)` (a real array): NumPy automatically promotes it to complex and adds element by element. The final result is an array of `dtype=complex128` whose every element is exactly `cos(theta[i]) + i·sin(theta[i])`.

In other words: whether `theta` is a single number or a whole array, the line `np.cos(theta) + 1j * np.sin(theta)` stays exactly the same — that's the payoff of NumPy broadcasting, with no hand-written loop. The little experiment below lets you watch this broadcasting happen:

In [ ]:
# Broadcasting mini-experiment: what happens when 1j multiplies an array?
import numpy as np

demo_theta = np.array([0, np.pi/2, np.pi])   # an array of 3 angles
demo_sin = np.sin(demo_theta)
demo_result = 1j * demo_sin                   # the scalar 1j broadcasts to every element

print('demo_theta    =', demo_theta)
print('demo_sin      =', demo_sin)
print('1j * demo_sin =', demo_result)
print('result dtype:', demo_result.dtype, '| shape:', demo_result.shape)

# add the cos part and confirm the structure matches euler's formula exactly
demo_cos = np.cos(demo_theta)
demo_full = demo_cos + demo_result
print('\ncos(theta) + 1j*sin(theta) =', demo_full)
print('✅ The scalar 1j broadcasts to every position in the array automatically; the code stays the same when theta is an array.')


In [ ]:
def euler(theta):
    # ✏️ TODO: return cosθ + i·sinθ
    raise NotImplementedError("TODO: implement euler(theta) — return cos(theta) + 1j*sin(theta)")


## 3. From the unit circle to the FFT twiddle factor

`e^{iθ}` means "turn to the position at angle θ". What the FFT needs is a set of regular rotations:

- `N`: how many slices one full turn is divided into
- `n`: the current sample index
- `k`: the current frequency index
- `-2π·k·n/N`: the angle this frequency must turn to at this sample

So `twiddle(k, n, N)` simply feeds this angle into Euler's formula.


### 3.1 ✏️ Task two: the FFT twiddle factor `twiddle`

`W_N^{kn} = e^{-2πi·k·n/N}`

**Line of reasoning**:
1. The twiddle factor's angle is `-2π·k·n/N`; **why must the sign be negative?**
   - From the conjugate point of view: the DFT analyzes the signal's **frequency content**, which amounts to asking in the complex plane "how big is the signal's projection onto the direction e^{+iωt}?". To extract the positive frequency we multiply by its conjugate e^{-iωt}, so components turning the same way reinforce and the others cancel.
   - From the point of view of mathematical habit: the inverse DFT (IDFT) uses `e^{+2πikn/N}`, so that the two undo each other. The forward transform takes the minus sign, the inverse the plus sign — a symmetric pair.
   - From the point of view of rotation direction: the minus sign means **clockwise rotation** (in the usual math coordinate system, counterclockwise is positive). It's a convention, but once chosen the whole FFT ecosystem is built on that direction.
2. Feed this angle into `euler(theta)` — no need to rederive cos/sin.
3. Prefer calling `euler` over `np.exp`: if you ever swap the underlying implementation, changing `euler` in one place takes effect everywhere.

**Reference input and output**: `twiddle(k=1, n=2, N=8)` = `euler(-π/2)` = 0-1j

> 💡 Need a hint? After finishing the exercise you can consult the reference implementation in the `solutions/` folder.

In [ ]:
def twiddle(k, n, N):
    # ✏️ TODO: return e^{-2πi·k·n/N}
    raise NotImplementedError("TODO: implement twiddle(k, n, N) — return euler(-2*np.pi*k*n/N)")


## 4. Check + print the points on the unit circle

In [ ]:
theta = np.linspace(0, 2*np.pi, 9)   # 0..360°, 9 points
z = euler(theta)
assert np.max(np.abs(z - np.exp(1j*theta))) < 1e-12, 'euler should equal np.exp(1j*theta)'
assert np.allclose(np.abs(z), 1.0), 'e^{iθ} always has magnitude 1 (it lies on the unit circle)'
assert abs(twiddle(0, 0, 8) - 1) < 1e-12, 'twiddle factor should be 1 when k·n=0'
assert abs(twiddle(1, 2, 8) - (0-1j)) < 1e-12, 'twiddle(1,2,8) should be -1j (non-trivial check)'
for ang, val in zip(np.degrees(theta), z):
    print(f'{ang:6.0f}° | {val.real:+.3f} {val.imag:+.3f}i')
print('\n✅ Complex exponential = rotation; twiddle = the FFT\'s twiddle factor.')

### Aside: why is `twiddle(1, 2, 8) = -1j` called a "non-trivial check"?

The assertions above contain two checks that look very similar:

```python
assert abs(twiddle(0, 0, 8) - 1) < 1e-12, 'twiddle factor should be 1 when k·n=0'
assert abs(twiddle(1, 2, 8) - (0-1j)) < 1e-12, 'twiddle(1,2,8) should be -1j (non-trivial check)'
```

The first one is actually pretty "trivial": the angle is `-2π·0·0/8 = 0`, and as long as the code has no silly mistake, `euler(0)` is almost certain to equal 1 (whether or not you understand Euler's formula, a throwaway placeholder implementation could easily pass by luck). This line only tests "nothing is obviously broken"; it can't tell whether you really understood the structure of the formula.

The second one, `twiddle(1, 2, 8) = -1j`, is the test that actually carries information, for three reasons:

1. **It lands exactly on a multiple of 90°**: angle = `-2π·1·2/8 = -π/2` (that is, -90°). The exact cos/sin values at such angles are known (tidy numbers like 0, +1, -1), so you need no calculator, yet it checks that both the cos and the sin branches are right — `cos(-π/2)=0` and `sin(-π/2)=-1` must both be correct to give `0 + (-1)i = -1j`.
2. **It catches subtle mistakes like a flipped sign or a misplaced term**: if the implementation drops the minus sign, or swaps where `k*n` and `N` go, a test with a random angle produces numbers that "look plausible" and can slip through; but test with a key angle like `-π/2` and any mistake lands visibly away from `-1j` (say `+1j` or `1+0j`), exposing the error at once.
3. **It's one of the most common "standard building blocks" in the FFT butterfly**: twiddle factors at multiples of 90° (1, -1j, -1, 1j) show up again and again in the radix-2 FFT butterfly. Verifying this special case cleanly today will save you plenty of debugging time when you hand-write the FFT in L37–L39.

In one sentence: **`twiddle(0,0,8)=1` only tests "nothing broke"; `twiddle(1,2,8)=-1j` tests "Euler's formula and the direction of the minus sign are really used correctly" — and that's why it's called a "non-trivial check".**

## 5. Draw the unit circle

In [ ]:
import matplotlib.pyplot as plt
fine = euler(np.linspace(0, 2*np.pi, 200))
plt.figure(figsize=(4.5, 4.5))
plt.plot(fine.real, fine.imag, '-', lw=1)
plt.plot(z.real, z.imag, 'o')
plt.gca().set_aspect('equal')
plt.title('e^{iθ} traces the unit circle')
plt.xlabel('Re'); plt.ylabel('Im'); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 📊 Figure: what is aviz.twiddles(8)?

The line of code below plots a visualization of the FFT twiddle-factor matrix — all 8×8 twiddle factors produced by every (k, n) pair:

- **Row index**: frequency k (from 0 to 7)
- **Column index**: time n (from 0 to 7)
- **The color of each cell**: the position of the complex number `twiddle(k, n, 8)` on the unit circle (shown by color and direction)

This figure shows directly "where each twiddle factor of the FFT matrix sits on the unit circle". You'll see:
- Row k=0 (the DC component) is all 1s, because `twiddle(0, n, 8) = e^0 = 1`
- The other rows follow a periodic pattern, rotating faster as k grows

These are the "building blocks" that the FFT implementation in L37–L39 will call over and over.

**🎉 When you're done**: save the notebook (`Ctrl+S`), then commit and push this lesson's changes following the repo's conventions.

## 🎨 Figure: 8 twiddle factors = the FFT's building blocks

In [ ]:
import aviz as aviz; aviz.style()
aviz.twiddles(8);

In [ ]:
# ✏️ Comparison: np.exp(1j*theta) vs cos+sin assembled by hand
# The two give exactly the same numbers, but which one is more "transparent"?

import numpy as np

print("Check: the two forms give exactly the same numbers\n")
angles = np.linspace(0, 2*np.pi, 5)
for theta in angles:
    z_exp = np.exp(1j*theta)
    z_manual = np.cos(theta) + 1j*np.sin(theta)
    diff = np.abs(z_exp - z_manual)
    print(f'theta={theta:.2f} | exp={z_exp:.2f} | cos+i sin={z_manual:.2f} | diff={diff:.1e}')

print("\n✅ The difference is within floating-point precision (< 1e-15); the two forms are equivalent.")
print("\nSo why assemble cos+sin by hand?")
print("- np.exp(1j*theta): concise, but a black box — Euler's formula's structure is hidden")
print("- cos+sin by hand: the real and imaginary parts appear explicitly, demonstrating e^{iθ} = cos θ + i·sin θ")
print("- In code review and teaching, the latter shows an understanding of 'why', not just 'what was used'.")

## Parameter experiment: turn one knob at a time

Compute `np.array([twiddle(1, n, 8) for n in range(8)])` to get 8 twiddle factors, print their phases (angles), and confirm they come out as 0°, -45°, -90°, -135°, 180°, 135°, 90°, 45° — eight points evenly spaced around the unit circle.

Change `N` from 8 to 16: the angular gap between neighboring twiddle factors halves (-22.5°), and the points on the unit circle become twice as dense. Change `k` from 1 to 3: each step now turns -135° — the FFT's frequency index directly controls the rotation speed.

### Aside: what the parameters in twiddle(k, n, N) mean

In the DFT formula `X[k] = Σ x[n] · e^{-2πikn/N}`, each of the three parameters of the twiddle factor `e^{-2πikn/N}` has its own job:

| Parameter | Range | Meaning | Role in the rotation |
|---|---|---|---|
| **k** | 0 to N-1 | **Frequency index**: which frequency component we're analyzing (0 Hz, Δf, 2Δf, …) | Controls the **rotation speed**: the bigger k, the faster the twiddle factor turns |
| **n** | 0 to N-1 | **Time sample index**: which sample (t=0, Δt, 2Δt, …) | Controls the **number of steps**: the bigger n, the more turns already made |
| **N** | constant | **Total number of samples**: how many samples the stretch of signal contains | Controls the **division of the circle**: the bigger N, the more slices one turn is cut into, and the smaller the angular-frequency gap between neighboring k |

**Numeric angle = -2π·k·n/N**:
- The numerator `k·n` is the product of two small indices; every (k, n) pair gives a different angle
- The denominator `N` cuts one turn into N slices, so `-2π/N` is the basic angular step
- The minus sign is a convention (see Task two above)

**Example**: say N=8 (8 samples) and k=1 (the 1st frequency):
- n=0: angle = -2π·1·0/8 = 0° (the hand points at 12 o'clock)
- n=1: angle = -2π·1·1/8 = -45° (turned 45° clockwise)
- n=2: angle = -2π·1·2/8 = -90° (turned 90° clockwise)
- ……
- n=7: angle = -2π·1·7/8 = -315° (turned 315° clockwise, the same as 45° counterclockwise)

The experiment below changes these three parameters so you can watch how the twiddle factor responds.

In [ ]:
# ✏️ Parameter experiment: turn one knob at a time
import numpy as np

# --- Experiment A: fix k=1, N=8, sweep n=0..7 ---
N = 8
factors = np.array([twiddle(1, n, N) for n in range(N)])
phases_deg = np.degrees(np.angle(factors))
print('Phase of each twiddle factor for N=8, k=1 (degrees):')
for n, (f, p) in enumerate(zip(factors, phases_deg)):
    print(f'  n={n}: {f.real:+.3f}{f.imag:+.3f}j  ∠{p:+.1f}°')

# --- Experiment B: change to N=16 and watch the angular gap halve ---
N2 = 16
factors2 = np.array([twiddle(1, n, N2) for n in range(N2)])
phases2 = np.degrees(np.angle(factors2))
print(f'\nN={N2}, k=1 : angular gap between neighboring twiddle factors: {abs(phases2[1]-phases2[0]):.1f}° (45° for N=8 — halved)')

## Wrap-up

You can now call `euler(theta)` to produce the complex number at any angle on the unit circle, and call `twiddle(k, n, N)` to compute the twiddle factors the FFT needs. These two functions correspond to the low-level calls behind the L37–L39 DFT implementation in `src/aurora/audio/transforms.py`. The next lesson (L36) weights the signal with a window function, and the windowed signal is ultimately fed into the `twiddle` you wrote today.

This section covers the same formula as `notebooks/1_complex_trig/L06_euler.ipynb`, but **with a different job**:

| L06 | L35 |
|---|---|
| Focus: **the mathematical intuition of Euler's formula** | Focus: **implementing Euler's formula in code** |
| Content: unit-circle pictures, complex-plane geometry, the polar view | Content: Python code, FFT parameters, twiddle-factor checks |
| Goal: why e^{iθ} = cos θ + i·sin θ | Goal: how to build twiddle factors in code |

- **If you've already done L06**: the "why Euler's formula holds" part of this lesson is review; the focus shifts to "how to implement it in code".
- **If you haven't done L06 yet**: after finishing this section, go back to L06's unit-circle pictures and complex-plane geometry — the two will confirm each other and deepen your understanding.

---
⬇️ **Final check**: you've read the wrap-up; now complete the whiteboard challenge below before ticking the self-review.


## ✏️ Whiteboard challenge: Euler's formula and twiddle factors by hand (target: 10 minutes)

Cover the screen and answer on paper:

**Question 1**: Work out three special values of `euler(θ)` by hand (no calculator):
- euler(0) = ?
- euler(π/2) = ?
- euler(π) = ?

**Question 2**: Verify the magnitude property: `|euler(θ)|` equals ? for any θ
(Use Pythagoras: √(cos²θ + sin²θ) = ?)

**Question 3**: The twiddle factor `twiddle(k=1, n=2, N=8)` = ?
- angle = -2π·1·2/8 = ? (radians)
- cos(?) + i·sin(?) = ? (exact value)

**Question 4**: What is `twiddle(k=0, n=n, N=N)` for any n and N? (Why?)

**Question 5**: For N=8, what pattern do `twiddle(1, n, 8)` for n=0,1,2,...,7 form on the unit circle?

When you've finished the derivation, run the cell below to check your answers.

In [ ]:
# ✏️ Answer-check cell
import numpy as np

# Q1: special values of euler
try:
    e0   = euler(0)
    e90  = euler(np.pi/2)
    e180 = euler(np.pi)
    assert np.isclose(e0,   1+0j,  atol=1e-12), f"euler(0) should be 1+0j, got {e0}"
    assert np.isclose(e90,  0+1j,  atol=1e-12), f"euler(π/2) should be 0+1j, got {e90}"
    assert np.isclose(e180, -1+0j, atol=1e-12), f"euler(π) should be -1+0j, got {e180}"
    print(f"Q1 ✅  euler(0)={e0}  euler(π/2)={e90}  euler(π)={e180}")
except (NotImplementedError, TypeError):
    print("⬜ Q1: implement euler() first, then run this answer-check cell")

# Q2: magnitude is always 1
thetas = np.linspace(0, 2*np.pi, 1000)
mags = np.abs(np.cos(thetas) + 1j*np.sin(thetas))
assert np.allclose(mags, 1.0, atol=1e-12)
print(f"Q2 ✅  |euler(θ)| = 1.0 holds for every θ (unit-circle property)")

# Q3: twiddle(1, 2, 8) = e^{-πi/2} = 0 - 1j
angle_312 = -2 * np.pi * 1 * 2 / 8  # = -π/2
expected_312 = np.cos(angle_312) + 1j*np.sin(angle_312)
assert np.isclose(expected_312, 0-1j, atol=1e-12), f"twiddle(1,2,8) should be 0-1j, got {expected_312}"
try:
    tw = twiddle(1, 2, 8)
    assert np.isclose(tw, 0-1j, atol=1e-12)
    print(f"Q3 ✅  twiddle(1,2,8) = e^{{-πi/2}} = {tw}")
except (NotImplementedError, TypeError):
    print("⬜ Q3: implement twiddle() first, then run this answer-check cell")

# Q4: twiddle(0, n, N) = e^0 = 1 for every n
try:
    for n_val in range(8):
        assert np.isclose(twiddle(0, n_val, 8), 1+0j, atol=1e-12)
    print(f"Q4 ✅  twiddle(0,n,N)=e^0=1 for every n (k=0 is the DC component; its twiddle factor is 1)")
except (NotImplementedError, TypeError):
    print("⬜ Q4: implement twiddle() first, then run this answer-check cell")

# Q5: the N=8 twiddle factors are spread evenly around the unit circle
angles_8 = np.array([-2*np.pi*1*n/8 for n in range(8)])
tw_8 = np.cos(angles_8) + 1j*np.sin(angles_8)
assert np.allclose(np.abs(tw_8), 1.0, atol=1e-12)  # all on the unit circle
# Neighboring phase difference is fixed at -π/4. Note that np.angle folds its output into (-π, π],
# so the differences must be wrapped back into the same range before comparing (otherwise the jump at ±π gives +7π/4).
diffs = np.diff(np.angle(tw_8))
diffs_wrapped = (diffs + np.pi) % (2 * np.pi) - np.pi
assert np.allclose(diffs_wrapped, -np.pi/4, atol=1e-12)
print(f"Q5 ✅  the 8 twiddle factors are evenly spaced, turning -π/4={np.degrees(-np.pi/4):.1f}° per step")
print("\n🎉 Euler whiteboard challenge passed! The twiddle factor e^{-2πikn/N} is the FFT's core building block.")

## Aside: what negative angles mean on the unit circle

The whiteboard challenge involves negative angles (such as -π/2). **The minus sign only indicates the direction of rotation**:
- **Positive angle** (e.g. π/4): counterclockwise
- **Negative angle** (e.g. -π/4): clockwise

The same point can be described by several angles:
- -π/2 (90° clockwise) = 3π/2 (270° counterclockwise) = the same point, at the bottom of the unit circle (0, -1)
- cos(-π/2) = cos(3π/2) = 0
- sin(-π/2) = sin(3π/2) = -1

**Remember the symmetries of the trig functions**:
- cos(-θ) = cos(θ) (cosine is an even function)
- sin(-θ) = -sin(θ) (sine is an odd function)

So when you work out twiddle(1, 2, 8) = euler(-π/2) by hand:
- cos(-π/2) = cos(π/2) = 0 ✓
- sin(-π/2) = -sin(π/2) = -1 ✓
- result = 0 + (-1)i = -1j ✓

In [ ]:
# ✏️ Self-review for this lesson
l35_review = {
    "euler_formula":          None,  # remember e^{iθ}=cosθ+i·sinθ? True/False
    "euler_implemented":      None,  # euler(theta) implemented and assertions pass? True/False
    "twiddle_implemented":    None,  # twiddle(k,n,N) implemented and assertions pass? True/False
    "unit_circle_property":   None,  # understand |e^{iθ}|=1 (the unit circle)? True/False
    "whiteboard_passed":      None,  # whiteboard challenge done on paper? True/False
}

unfilled = [k for k, v in l35_review.items() if v is None]
assert not unfilled, f'not filled in yet: {unfilled}'
weak = [k for k, v in l35_review.items() if v is False]
if weak:
    print(f'⚠️  needs more work: {weak}')
else:
    print('✅ L35 fully cleared! On to L36: window functions')

---

→ **Next lesson** · [L36 · Window functions](https://github.com/djzoom/AURORA/blob/main/notebooks/5_audio_dsp/L36_windows.ipynb) *(Chinese)*

> Next lesson covers **window functions**: spectral leakage from the rectangular window; Hann / Hamming / Blackman compared.